# 工期不稳定时，怎样估计延期概率与安排缓冲？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'deadline': 16.174379145417845, 'selected_on_development': 'end', 'development': {'end': {'late_probability': 0.471, 'mean_finish': 16.149741325248, 'missed_booking_rate': 0.471}, 'before': {'late_probability': 0.492, 'mean_finish': 16.46304327325856, 'missed_booking_rate': 0.079}}, 'independent_simulation': {'end': {'late_probability': 0.4595, 'mean_finish': 16.13339112782704, 'p90_finish': 20.106908841698377, 'missed_booking_rate': 0.46}, 'before': {'late_probability': 0.46, 'mean_finish': 16.5573198637733, 'p90_finish': 19.526055838271578, 'missed_booking_rate': 0.103}}, 'simulation_only': True, 'before_minus_end_late_probability': {'estimate': 0.0005, 'approx_95_monte_carlo_interval': [-0.02113211256320042, 0.02213211256320042], 'excludes_model_uncertainty': True}}


## 1. 补全：核对延期率
从 finish_by_policy 重算选中方案延期比例，并与指标核对。

In [2]:
rate=float(np.mean(finish_by_policy[selected]>deadline))
assert np.isclose(rate,future_results[selected]['late_probability'])
print(rate)

0.4595


## 2. 修改：承诺日期晚两天
保持模拟结果不变，仅将交付承诺推迟两天；延期率不应上升。

In [3]:
later={name:float(np.mean(finish>deadline+2)) for name,finish in finish_by_policy.items()}
assert all(later[name]<=future_results[name]['late_probability'] for name in policies)
print(later)

{'end': 0.2915, 'before': 0.1075}


## 3. 迁移：没有预约限制的小项目
去掉预约时点与重新预约等待，按纯依赖计算完工时间，验证不晚于受预约约束的结果。

In [4]:
ready=future[:,0]+np.maximum(future[:,1],future[:,2])
unconstrained=np.maximum(ready+future[:,3]+future[:,4],future[:,0]+future[:,5])
assert np.all(unconstrained<=finish_by_policy['end']+1e-10)
print({'无预约约束平均工期':float(unconstrained.mean()),'延期率':float(np.mean(unconstrained>deadline))})

{'无预约约束平均工期': 14.105804765808852, '延期率': 0.174}
